# Lab 3 — Simple Tool-Using Agent

**Student:** Sharon Weisman  
**Teammate:** Rom Ben Yakar

## Goal
Implement a simple agent that uses a single tool in this loop:

**User input → Agent → Tool call → Tool result → Explanation**

In this solution, the agent receives a log/event message, calls a tool named `analyze_log()`, and returns a structured explanation including severity, category, matched patterns, and recommended action.


## 1. Tool implementation

The tool is responsible for analyzing the log text and returning a structured result.
It uses simple rule-based logic to detect patterns such as failed logins, malware indicators, unauthorized access, timeouts, and application errors.


In [ ]:
from typing import Dict, List

def recommend_action(severity: str) -> str:
    mapping = {
        "Critical": "Escalate immediately, isolate affected assets, and begin incident response.",
        "High": "Investigate promptly, validate scope, and notify the relevant security or operations team.",
        "Medium": "Review logs, monitor the system, and create a follow-up task if needed.",
        "Low": "Record the event and continue monitoring.",
    }
    return mapping.get(severity, "Review the event manually.")


def analyze_log(log_text: str) -> Dict[str, object]:
    text = log_text.lower()

    matched_patterns: List[str] = []
    severity = "Low"
    category = "General Event"
    explanation = "The event does not currently indicate a critical security issue."

    if "failed login" in text or "failed authentication" in text:
        matched_patterns.append("failed login")
        severity = "High"
        category = "Authentication Attack"
        explanation = (
            "Multiple failed authentication attempts may indicate a brute-force "
            "attack or unauthorized access attempt."
        )

    if "unauthorized" in text or "forbidden" in text or "access denied" in text:
        matched_patterns.append("unauthorized access")
        if severity not in ["Critical", "High"]:
            severity = "High"
        category = "Access Violation"
        explanation = (
            "The event suggests an unauthorized or blocked access attempt that "
            "should be investigated."
        )

    if "malware" in text or "trojan" in text or "ransomware" in text:
        matched_patterns.append("malware indicator")
        severity = "Critical"
        category = "Malware Activity"
        explanation = (
            "The log contains malware-related indicators, which may represent "
            "active compromise and require immediate response."
        )

    if "data exfiltration" in text or "large outbound transfer" in text:
        matched_patterns.append("exfiltration indicator")
        severity = "Critical"
        category = "Data Exfiltration"
        explanation = (
            "The event may indicate suspicious outbound data transfer and possible "
            "data exfiltration."
        )

    if "timeout" in text or "timed out" in text:
        matched_patterns.append("timeout")
        if severity == "Low":
            severity = "Medium"
        if category == "General Event":
            category = "Service Reliability"
        explanation = (
            "The event indicates a timeout condition. This may reflect service "
            "degradation, connectivity issues, or temporary overload."
        )

    if "error" in text or "exception" in text or "crash" in text:
        matched_patterns.append("application error")
        if severity == "Low":
            severity = "Medium"
        if category == "General Event":
            category = "Application Failure"
        explanation = (
            "The event shows an application or service error. This may affect "
            "availability and should be reviewed."
        )

    if not matched_patterns:
        matched_patterns.append("no known high-risk pattern")

    return {
        "severity": severity,
        "category": category,
        "matched_patterns": matched_patterns,
        "explanation": explanation,
        "recommended_action": recommend_action(severity),
    }

## 2. Agent implementation

The agent receives the user input, calls the tool, reads the structured tool result, and returns a final explanation.


In [ ]:
def agent(user_input: str) -> str:
    tool_result = analyze_log(user_input)

    response = f"""
=== Agent Analysis Report ===
Input log:
{user_input}

Tool called:
analyze_log()

Detected category:
{tool_result['category']}

Severity:
{tool_result['severity']}

Matched patterns:
{", ".join(tool_result['matched_patterns'])}

Explanation:
{tool_result['explanation']}

Recommended action:
{tool_result['recommended_action']}
""".strip()

    return response

## 3. Test examples

Below are several test cases to demonstrate the agent loop in action.


In [ ]:
sample_logs = [
    "Failed login attempt from IP 192.168.1.10",
    "Service timeout while connecting to database",
    "Unauthorized access to admin panel",
    "Possible malware detected on endpoint",
    "Application error in payment service",
    "Large outbound transfer detected from finance server",
]

for log in sample_logs:
    print(agent(log))
    print("\n" + "-" * 70 + "\n")

## 4. Design explanation

This solution follows the exact lab requirement of implementing a simple agent with one tool.

### Flow
1. The user provides a log message.
2. The agent calls `analyze_log()`.
3. The tool classifies the event.
4. The agent produces a readable explanation.

### Why this is a good solution
- It clearly demonstrates the **agent → tool → explanation** loop.
- The tool returns **structured data**, not just free text.
- The agent separates **analysis logic** from **response generation**.
- The solution is simple, readable, and easy to extend later.

### Possible future improvements
- Add more tools such as `check_ip_reputation()` or `lookup_mitre_mapping()`
- Return JSON output
- Connect the agent to a real log source
- Add user input interaction


## 5. Conclusion

This lab demonstrates a basic tool-using agent for cybersecurity log analysis.  
The implementation shows how an agent can receive user input, call a single tool, interpret the result, and generate a final explanation.  
This structure is simple but important because it forms the basis for more advanced AI security agents.
